In [38]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


from dotenv import load_dotenv
load_dotenv()

from google import genai
key = os.getenv("ai_key")

client = genai.Client(api_key=key)

In [39]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stackoverflow/pythonquestions")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pythonquestions' dataset.
Path to dataset files: /kaggle/input/pythonquestions


In [40]:
questions = pd.read_csv(os.path.join(path, "Questions.csv"), encoding="latin1")
answers   = pd.read_csv(os.path.join(path, "Answers.csv"), encoding="latin1")
tags      = pd.read_csv(os.path.join(path, "Tags.csv"), encoding="latin1")

In [41]:
questions.head()

,Id,OwnerUserId,CreationDate,Score,Title,Body
0,469,147.0,2008-08-02T15:11:16Z,21,How can I find the full path to a font from it...,<p>I am using the Photoshop's javascript API t...
1,502,147.0,2008-08-02T17:01:58Z,27,Get a preview JPEG of a PDF on Windows?,<p>I have a cross-platform (Python) applicatio...
2,535,154.0,2008-08-02T18:43:54Z,40,Continuous Integration System for a Python Cod...,<p>I'm starting work on a hobby project with a...
3,594,116.0,2008-08-03T01:15:08Z,25,cx_Oracle: How do I iterate over a result set?,<p>There are several ways to iterate over a re...
4,683,199.0,2008-08-03T13:19:16Z,28,Using 'in' to match an attribute of Python obj...,<p>I don't remember whether I was dreaming or ...


In [42]:
questions.info()
answers.info()
tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607282 entries, 0 to 607281
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Id            607282 non-null  int64  
 1   OwnerUserId   601070 non-null  float64
 2   CreationDate  607282 non-null  object 
 3   Score         607282 non-null  int64  
 4   Title         607282 non-null  object 
 5   Body          607282 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 27.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987122 entries, 0 to 987121
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Id            987122 non-null  int64  
 1   OwnerUserId   981755 non-null  float64
 2   CreationDate  987122 non-null  object 
 3   ParentId      987122 non-null  int64  
 4   Score         987122 non-null  int64  
 5   Body          987122 non-null  object 
dtypes: float

In [43]:
questions.describe()

,Id,OwnerUserId,Score
count,6.072820e+05,6.010700e+05,607282.000000
mean,2.371960e+07,2.519595e+06,2.283137
std,1.124715e+07,1.910375e+06,19.285578
min,4.690000e+02,2.500000e+01,-44.000000
25%,1.485519e+07,8.539340e+05,0.000000
50%,2.531897e+07,2.107677e+06,1.000000
75%,3.358823e+07,3.991164e+06,2.000000
max,4.014336e+07,7.044992e+06,5524.000000


In [44]:
questions = questions[questions["Score"] > 0]

In [45]:
answers.describe()

,Id,OwnerUserId,ParentId,Score
count,9.871220e+05,9.817550e+05,9.871220e+05,987122.000000
mean,2.160744e+07,1.589642e+06,2.053496e+07,3.028437
std,1.179488e+07,1.649346e+06,1.195486e+07,21.263246
min,4.970000e+02,1.000000e+00,4.690000e+02,-38.000000
25%,1.144775e+07,2.603030e+05,9.968532e+06,0.000000
50%,2.254188e+07,9.634650e+05,2.103550e+07,1.000000
75%,3.205469e+07,2.433810e+06,3.115233e+07,3.000000
max,4.014337e+07,7.044747e+06,4.014319e+07,8384.000000


In [46]:
answers = answers[answers["Score"] > 0].copy()

In [47]:
# 1. Compute the mean score for each question
mean_scores = answers.groupby("ParentId")["Score"].mean().reset_index()
mean_scores.rename(columns={"Score": "mean_score"}, inplace=True)

# 2. Attach the mean score to each answer
answers_with_mean = answers.merge(
    mean_scores,
    on="ParentId",
    how="left"
)

# 3. Keep only answers with score > mean score of that question
answers_filtered = answers_with_mean[
    answers_with_mean["Score"] > answers_with_mean["mean_score"]
]

# 4. Group only the filtered answers
#answers_grouped = answers_filtered.groupby("ParentId")["Body"].apply(list).reset_index()
answers_grouped = (
    answers_filtered
    .groupby("ParentId")
    .apply(
        lambda g: [
            {
                "body": row["Body"],
                "score": row["Score"]
            }
            for _, row in g.iterrows()
        ]
    )
    .reset_index(name="answers_list")
)

answers_grouped.head()

/tmp/ipykernel_1887/2160633413.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,ParentId,answers_list
0,469,[{'body': '<p>Unfortunately the only API that ...
1,502,[{'body': '<p>ImageMagick delegates the PDF->b...
2,535,[{'body': '<p>One possibility is Hudson. It's...
3,594,[{'body': '<p>The canonical way is to use the ...
4,683,[{'body': '<p>Are you looking to get a list of...


In [48]:
# 5. Merge questions with the filtered answers
df = questions.merge(
    answers_grouped,
    left_on="Id",
    right_on="ParentId",
    how="left"
)

# 6. Replace NaN with empty lists for questions without good answers
df["answers_list"] = df["answers_list"].apply(lambda x: x if isinstance(x, list) else [])

# 7. Rename columns for clarity
df.rename(columns={
    "Body_x": "question_body",
}, inplace=True)

In [49]:
df.head()

,Id,OwnerUserId,CreationDate,Score,Title,Body,ParentId,answers_list
0,469,147.0,2008-08-02T15:11:16Z,21,How can I find the full path to a font from it...,<p>I am using the Photoshop's javascript API t...,469.0,[{'body': '<p>Unfortunately the only API that ...
1,502,147.0,2008-08-02T17:01:58Z,27,Get a preview JPEG of a PDF on Windows?,<p>I have a cross-platform (Python) applicatio...,502.0,[{'body': '<p>ImageMagick delegates the PDF->b...
2,535,154.0,2008-08-02T18:43:54Z,40,Continuous Integration System for a Python Cod...,<p>I'm starting work on a hobby project with a...,535.0,[{'body': '<p>One possibility is Hudson. It's...
3,594,116.0,2008-08-03T01:15:08Z,25,cx_Oracle: How do I iterate over a result set?,<p>There are several ways to iterate over a re...,594.0,[{'body': '<p>The canonical way is to use the ...
4,683,199.0,2008-08-03T13:19:16Z,28,Using 'in' to match an attribute of Python obj...,<p>I don't remember whether I was dreaming or ...,683.0,[{'body': '<p>Are you looking to get a list of...


In [50]:
df = df[["Id", "Title", "Body","Score", "answers_list"]]
df.head()

,Id,Title,Body,Score,answers_list
0,469,How can I find the full path to a font from it...,<p>I am using the Photoshop's javascript API t...,21,[{'body': '<p>Unfortunately the only API that ...
1,502,Get a preview JPEG of a PDF on Windows?,<p>I have a cross-platform (Python) applicatio...,27,[{'body': '<p>ImageMagick delegates the PDF->b...
2,535,Continuous Integration System for a Python Cod...,<p>I'm starting work on a hobby project with a...,40,[{'body': '<p>One possibility is Hudson. It's...
3,594,cx_Oracle: How do I iterate over a result set?,<p>There are several ways to iterate over a re...,25,[{'body': '<p>The canonical way is to use the ...
4,683,Using 'in' to match an attribute of Python obj...,<p>I don't remember whether I was dreaming or ...,28,[{'body': '<p>Are you looking to get a list of...


In [51]:
def build_document(row):
    question_score = row.get("Score", "")
    answers = row.get("answers_list") or []

    formatted_answers = []
    for i, ans in enumerate(answers, start=1):
        formatted_answers.append(
            f"Answer {i} - Score: {ans['score']}\n{ans['body']}"
        )

    answers_text = "\n\n".join(formatted_answers)

    return (
        f"Title: {row.get('Title', '')}\n"
        f"Question Score: {question_score}\n\n"
        f"Question:\n{row.get('Body', '')}\n\n"
        f"Answers:\n{answers_text}"
    )


df["document"] = df.apply(build_document, axis=1)

In [52]:
print(df.document.iloc[50])

Title: Find broken symlinks with Python
Question Score: 13

Question:
<p>If I call <code>os.stat()</code> on a broken <code>symlink</code>, python throws an <code>OSError</code> exception. This makes it useful for finding them. However, there are a few other reasons that <code>os.stat()</code> might throw a similar exception. Is there a more precise way of detecting broken <code>symlinks</code> with Python under Linux?</p>


Answers:
Answer 1 - Score: 11
<p><a href="https://docs.python.org/2/library/os.html#os.lstat" rel="nofollow">os.lstat()</a> may be helpful. If lstat() succeeds and stat() fails, then it's probably a broken link.</p>


Answer 2 - Score: 18
<p>A common Python saying is that it's easier to ask forgiveness than permission.  While I'm not a fan of this statement in real life, it does apply in a lot of cases.  Usually you want to avoid code that chains two system calls on the same file, because you never know what will happen to the file in between your two calls in your

In [53]:
def clean(text):
    return BeautifulSoup(str(text), "html.parser").get_text()

df["document"] = df["document"].apply(clean)


/tmp/ipykernel_1887/117330258.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return BeautifulSoup(str(text), "html.parser").get_text()


In [54]:
print(df.document.iloc[0])

Title: How can I find the full path to a font from its display name on a Mac?
Question Score: 21

Question:
I am using the Photoshop's javascript API to find the fonts in a given PSD.
Given a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.
This is all happening in a python program running on OSX so I guess I'm looking for one of:

Some Photoshop javascript
A Python function
An OSX API that I can call from python



Answers:
Answer 1 - Score: 12
Unfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.
Cocoa doesn't have any native support, at least as of 10.5, for getting the location of a font.


In [55]:
def fix_surrogates(text):
    if isinstance(text, str):
        return text.encode("utf-8", "surrogatepass").decode("utf-8", "ignore")
    return text

df = df.applymap(fix_surrogates)

/tmp/ipykernel_1887/3173901628.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(fix_surrogates)


In [56]:
# Crear rutas
df_clean_path =  "df_clean.parquet"


# Guardar
df.to_parquet(df_clean_path)


# Leer
df = pd.read_parquet(df_clean_path)

In [57]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")


documents = list(df["document"].values)

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=512,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/641 [00:00<?, ?it/s]

In [58]:
"""
model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")

documents = df["document"].tolist()

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32
).astype("float32")

dim = embeddings.shape[1]
print(dim)
"""


'\nmodel = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")\n\ndocuments = df["document"].tolist()\n\nembeddings = model.encode(\n    documents,\n    convert_to_numpy=True,\n    show_progress_bar=True,\n    batch_size=32\n).astype("float32")\n\ndim = embeddings.shape[1]\nprint(dim)\n'

In [59]:
index = faiss.IndexFlatL2(dim)
index.add(embeddings)


In [60]:
def retrieve(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    distances, indices = index.search(q_emb, k)
    return df.loc[indices[0], ["document"]]

In [61]:
docs = retrieve("How can I open a CSV file in Python?")

In [62]:
for i, doc in enumerate(docs["document"]):
    print(f"\n===== DOCUMENT {i} =====\n")
    print(doc)


===== DOCUMENT 0 =====

Title: How to open .csv file from a url with Python?
Question Score: 1

Question:
I'm trying to open a csv file from a url but for some reason I get an error saying that there is an invalid mode or filename. I'm not sure what the issue is. Help?
url = "http://...."
data = open(url, "r")
read = csv.DictReader(data)



Answers:


===== DOCUMENT 1 =====

Title: Reading and Writing data in CSV files in Python
Question Score: 1

Question:
I am new to Python and I am having some problems with CSV files in Python. Please help me out

How do I open and read csv files in Python which are present in some other directory? I know we can do

import csv 
f = open('attendees1.csv')

As long as my program file and csv file is in the same directory. But how do I provide a link to the csv file sitting in another directory?
I have a list with multiple rows and columns, how do I transfer this data to a csv file and save it down in a particular location?

Please help me out


Answe

In [63]:
def build_context(rows):
    return "\n\n---\n\n".join(rows["document"].tolist())


In [64]:
def call_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [65]:
def rag_answer(query):
    retrieved = retrieve(query)

    # Construimos el contexto a partir de los documentos ya formateados
    context = build_context(retrieved)

    prompt = f"""
    You are an expert Python assistant. Answer ONLY using the information in the context.

    ---------------- CONTEXT START ----------------
    {context}
    ---------------- CONTEXT END ------------------

    QUESTION:
    {query}

    FINAL ANSWER (in English):
    """

    # Llamada al modelo
    answer = call_gemini(prompt)

    # Devolvemos ambos
    return {
        "prompt": prompt,
        "retrieved_docs": retrieved,
        "answer": answer
    }

In [66]:
result = rag_answer("How do I open a CSV in Python?")


print(result["answer"])

Based on the context, you can open a CSV file in Python using the `open()` function.

Examples provided in the context are:
*   `f = open('attendees1.csv')`
*   `with open('cancerdata.csv', 'rb') as csvfile:`
*   `_csvFile = open (_csvFilename, 'wb')`
*   `open('you_file.csv', 'rb')` (used within a `csv.reader` call)

The context also shows `import csv` is used when working with CSV files.


In [67]:
print(result["prompt"])


    You are an expert Python assistant. Answer ONLY using the information in the context.

    ---------------- CONTEXT START ----------------
    Title: How to open .csv file from a url with Python?
Question Score: 1

Question:
I'm trying to open a csv file from a url but for some reason I get an error saying that there is an invalid mode or filename. I'm not sure what the issue is. Help?
url = "http://...."
data = open(url, "r")
read = csv.DictReader(data)



Answers:


---

Title: How to open a csv file in Microsoft Excel in Python?
Question Score: 3

Question:
base_path = os.path.dirname(os.path.abspath(__file__))          
_csvFilename = os.path.join(base_path, "bcForecasting.csv")
_csvFile = open (_csvFilename, 'wb')
_csvFile = csv.writer(_csvFile, quoting=csv.QUOTE_ALL)

_Header = self.makeIntoList (self.root.tss.series () [0].getAllTimes (), self.originalTimesteps + _futurePeriods)
_csvFile.writerow (_Header)

Now I want to open the created bcForecasting.csv file in Excel. How

In [68]:
result = rag_answer("¿How can I open a CSV file in Python using Pandas?")

In [69]:
print(result["answer"])

To open a CSV file in Python using Pandas, you can use `pandas.read_csv`.

Examples from the context show its usage:
*   `pd.read_csv("sample.csv",sep="|")`
*   `df = pd.read_csv(io.StringIO(temp), sep="|", comment='-')`


In [70]:
result = rag_answer("Give me an example of Inheritance in Python")
print(result["answer"])

Let's say you have a class called `Animal`. In this `Animal` class, you can have a method called `walk` that prints "Animal is walking", and a variable like a `weight` field.

Now, imagine you create two other classes: `Bird` and `Monkey`. If `Bird` and `Monkey` "inherit" from `Animal`, it means they automatically get the `walk` method and the `weight` field. So, both a `Bird` and a `Monkey` can `walk`, and they both have a `weight`.

But `Bird` can also have an additional method like `fly` (which prints that a bird can fly) and a unique variable like `max_fly_altitude`. Similarly, `Monkey` can have its own additional method like `climb` (which prints that a monkey can climb trees) and a unique variable like `max_power`.

These unique features (`fly`, `max_fly_altitude` for Bird; `climb`, `max_power` for Monkey) are specific to those types of animals and are not part of the generic `Animal` class. The `Animal` class itself doesn't automatically gain the `fly` or `climb` features, becau

In [71]:
result = rag_answer("what are the best libraries for Machine Learning in Python")
print(result["answer"])

The context highly recommends `scikit-learn` as an equivalent to Java's Mahout for scalable machine learning, noting its fast and numerically-efficient algorithms and BSD license. It also features gradient boosted regression trees for classification and regression, and has binary installers available for Python 3.2.

Other Python AI libraries mentioned include `PyBrain`, `OpenCV`, `PyML`, and `PyEvolve`.

For Support Vector Machines (SVMs), `LibSVM` includes a Python wrapper. `vowpal wabbit`, `LaSVM` (via a ctypes wrapper), and `Elefant` are mentioned for online learning and online-SVM code. The "Programming Collective Intelligence" book also uses Python for its examples.


In [72]:
result = rag_answer("is Python a snake?")
print(result["answer"])

No, Python is not a snake.

Based on the context:
*   "Python" is the name of a programming language.
*   "Python" is described as "a set of rules (like syntax rules, or descriptions of standard functionality)".
*   Implementations of this language include CPython, Jython, IronPython, and PyPy.
*   One question explicitly discusses the challenge of differentiating articles about the "python language" from articles about the "snake" (the animal), indicating they are distinct.
*   Another question describes making a game "of snake" using Python, where "snake" refers to the game element (presumably an animal or representation thereof) and Python is the programming language used to create it.
